# Score a .h5 model on the paper's metrics

Metrics from Heikal et al. (Sci Rep 2024): accuracy, precision, recall, F1, ROC-AUC, binary cross-entropy loss, confusion matrix. Everything prints inline; nothing is saved to disk.

Test folder layout (first class name = label 0, second = positive label 1):
```
test_dir/
    benign/      *.png
    malignant/   *.png
```

In [ ]:
# ---- Config: edit these ----
MODEL_PATH = 'model.h5'
DATA_DIR = 'test_dir'
CLASS_NAMES = ['benign', 'malignant']  # [negative, positive]
IMG_SIZE = 224
BATCH_SIZE = 18
THRESHOLD = 0.5

# Set True only if the model does NOT contain its own Rescaling layer.
# (The paper's custom CNN has one built in.)
RESCALE = False

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    log_loss, precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay,
)

%matplotlib inline

In [ ]:
# compile=False: only inference is needed, avoids errors from custom optimizers/losses
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='binary',
    class_names=CLASS_NAMES,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,  # keeps predictions aligned with labels
)
if RESCALE:
    ds = ds.map(lambda x, y: (x / 255.0, y), num_parallel_calls=tf.data.AUTOTUNE)
ds = ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
# predict() runs in inference mode, so augmentation layers are inactive
y_true = np.concatenate([y.numpy().reshape(-1) for _, y in ds]).astype(int)
raw = np.asarray(model.predict(ds, verbose=1))

# Handles a single sigmoid output or a 2-unit softmax
y_prob = raw[:, 1] if (raw.ndim == 2 and raw.shape[1] == 2) else raw.reshape(-1)
y_pred = (y_prob >= THRESHOLD).astype(int)

assert len(np.unique(y_true)) == 2, 'Test set has only one class; ROC-AUC is undefined.'
print(f'{len(y_true)} samples, positive class = {CLASS_NAMES[1]}')

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

metrics = {
    'Accuracy': accuracy_score(y_true, y_pred),
    'Precision': precision_score(y_true, y_pred, zero_division=0),
    'Recall (sensitivity)': recall_score(y_true, y_pred, zero_division=0),
    'Specificity': tn / (tn + fp) if (tn + fp) else float('nan'),
    'F1-score': f1_score(y_true, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_true, y_prob),
    'Average precision': average_precision_score(y_true, y_prob),
    'Loss (binary cross-entropy)': log_loss(y_true, np.clip(y_prob, 1e-7, 1 - 1e-7), labels=[0, 1]),
}

print(f'=== Overall (threshold = {THRESHOLD}) ===')
for k, v in metrics.items():
    print(f'{k:30s} {v:.4f}')

print('\n=== Confusion matrix counts ===')
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')

print('\n=== Per class ===')
for idx, name in enumerate(CLASS_NAMES):
    p = precision_score(y_true, y_pred, pos_label=idx, zero_division=0)
    r = recall_score(y_true, y_pred, pos_label=idx, zero_division=0)
    f = f1_score(y_true, y_pred, pos_label=idx, zero_division=0)
    print(f'{name:12s} precision={p:.4f}  recall={r:.4f}  f1={f:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# ROC curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
axes[0].plot(fpr, tpr, label=f"AUC = {metrics['ROC-AUC']:.3f}")
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC curve')
axes[0].legend(loc='lower right')

# Precision-recall curve
prec, rec, _ = precision_recall_curve(y_true, y_prob)
axes[1].plot(rec, prec, label=f"AP = {metrics['Average precision']:.3f}")
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall curve')
axes[1].legend(loc='lower left')

# Confusion matrix
ConfusionMatrixDisplay(
    confusion_matrix(y_true, y_pred, labels=[0, 1]), display_labels=CLASS_NAMES
).plot(ax=axes[2], colorbar=False)
axes[2].set_title('Confusion matrix')

plt.tight_layout()
plt.show()